# Customer Churn Analysis using SQL

## Objective
To analyse customer churn behaviour and identify key factors contributing to customer attrition in a telecom company.

In [2]:
!pip install pandas


## Dataset Overview

The dataset contains customer information from a telecom company, including demographics, account details, and service usage.

Key variable:
- **Churn**: Indicates whether a customer has discontinued their service (Yes/No).

In [3]:
import pandas as pd

df = pd.read_csv("Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Analysis 1: Overall Churn Rate

We begin by calculating the overall churn rate to understand the proportion of customers who have left the service.

In [4]:
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('telco', conn, index=False, if_exists='replace')

7043

In [5]:
query = """
SELECT 
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS churn_rate
FROM telco;
"""

pd.read_sql(query, conn)

,total_customers,churned_customers,churn_rate
0,7043,1869,0.26537


### Key Insight

Approximately **26.5% of customers have churned**, indicating that over one-quarter of the customer base has discontinued their service. 

This suggests a significant customer retention issue and highlights the need to further investigate the key drivers of churn.

## Analysis 2: Churn by Contract Type

We analyse churn rates across different contract types to identify which groups of customers are more likely to leave.

In [8]:
query2 = """
SELECT 
    Contract, COUNT(*) AS total_customers
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS churn_rate
FROM telco
GROUP BY Contract;
"""

pd.read_sql(query, conn)

,total_customers,churned_customers,churn_rate
0,7043,1869,0.26537


In [9]:
query2 = """
SELECT 
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS churn_rate
FROM telco
GROUP BY Contract;
"""

pd.read_sql(query2, conn)

,Contract,total_customers,churned,churn_rate
0,Month-to-month,3875,1655,0.427097
1,One year,1473,166,0.112695
2,Two year,1695,48,0.028319


### Key Insight

Customers on **month-to-month contracts exhibit the highest churn rate at approximately 42.7%**, significantly higher than customers on longer-term contracts.

In contrast, customers on one-year and two-year contracts show substantially lower churn rates of 11.3% and 2.8% respectively.

This indicates that **contract length is a major driver of customer retention**, with longer commitments strongly associated with reduced likelihood of churn.

### Business Recommendation

The company should prioritise converting month-to-month customers into longer-term contracts through targeted incentives such as discounts or bundled offers.

Additionally, retention strategies should focus on customers early in their lifecycle, particularly those without long-term commitments, as they are significantly more likely to churn.

## Analysis 3: Churn by Customer Tenure

We analyse churn rates across different customer tenure groups to understand whether the length of time a customer has stayed with the company influences their likelihood of churning.

Customers are segmented into three groups:
- **New**: 0–12 months  
- **Mid-term**: 13–36 months  
- **Long-term**: 37+ months  

This segmentation allows us to identify whether newer customers are more at risk of leaving compared to more established customers.

In [20]:
query3 = """

SELECT 
    CASE
        WHEN tenure <= 12 THEN 'New'
        WHEN tenure <= 36 THEN 'Mid'
        ELSE 'Long'
    END AS tenure_group,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS churn_rate
FROM telco
GROUP BY 
    CASE
        WHEN tenure <= 12 THEN 'New'
        WHEN tenure <= 36 THEN 'Mid'
        ELSE 'Long'
    END;
"""

pd.read_sql(query3, conn)

,tenure_group,total_customers,churned,churn_rate
0,Long,3001,358,0.119294
1,Mid,1856,474,0.255388
2,New,2186,1037,0.474382


### Key Insight

Customers in the **New tenure group (0–12 months)** exhibit the highest churn rate at approximately **47.4%**, indicating that nearly half of newly acquired customers discontinue their service within the first year.

In contrast, customers in the **Long tenure group (37+ months)** show a significantly lower churn rate of approximately **11.9%**, suggesting that longer-term customers are much more stable and less likely to leave.

This highlights that **customer tenure is a critical driver of churn**, with early-stage customers being the most vulnerable segment.

### Business Recommendation

The company should prioritise retention strategies targeted at **new customers**, particularly within their first year of service. 

This may include onboarding support, personalised engagement, and early-stage incentives such as loyalty rewards or contract upgrade offers.

By strengthening engagement during the initial customer lifecycle, the company can reduce early churn and improve long-term customer retention.

## Analysis 4: Monthly Charges and Churn

We compare the average monthly charges between customers who churned and those who did not to assess whether pricing may influence churn behaviour.

In [23]:
query4 = """

SELECT 
    Churn,
    AVG(MonthlyCharges) AS avg_monthly_charges
FROM telco
GROUP BY Churn;
"""

pd.read_sql(query4, conn)


,Churn,avg_monthly_charges
0,No,61.265124
1,Yes,74.441332


### Key Insight

Customers who churned have a higher average monthly charge (74.44), compared to those who remained ($61.27).

This suggests that **higher pricing may be associated with increased likelihood of churn**, indicating that customers may be sensitive to cost when deciding whether to continue their service.

### Business Recommendation

The company should evaluate its pricing strategy, particularly for higher-paying customers, to ensure that perceived value aligns with cost.

Potential actions include offering value-added services, personalised plans, or targeted discounts to high-paying customers in order to reduce churn risk.